# evaluatorB — Primary temporal-localization scorer (ID-agnostic tIoU)

Notebook mirror of `evaluatorB.py` (same formulas, same numbers). Run all cells top-to-bottom.

**Protocol** (ActivityNet-style temporal matching, single interaction class):
1. Intervals are inclusive frame spans `[start, end]`.
2. Pairwise temporal IoU over the full GT × pred matrix (IDs ignored):
   `inter = max(0, min(e1,e2) − max(s1,s2) + 1)`, `union = dur_GT + dur_pred − inter`.
3. One-to-one assignment via Hungarian algorithm on `−tIoU`; zero-overlap pairs discarded.
4. Per threshold `t`: `TP` = matched pairs with `tIoU ≥ t`, `FP = n_pred − TP`, `FN = n_gt − TP` → standard P/R/F1.
   `mean_F1` = mean of F1 over thresholds (NOT mAP — the pipeline emits binary intervals with no confidence scores).

Pred summaries carry the pipeline's frame stitching (threshold=60, min-length=30, stride-5); GT does not, so ~1-stride boundary offsets are inherent.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
pd.options.display.float_format = "{:.2f}".format

CWD = Path.cwd()
if (CWD / "eval" / "gt_summary").exists():
    ROOT = CWD            # kernel launched from src/
elif (CWD / "gt_summary").exists():
    ROOT = CWD.parent     # kernel launched from src/eval/
else:
    raise SystemExit(f"Cannot locate gt_summary from {CWD}")

GT_DIR = ROOT / "eval" / "gt_summary"
PRED_DIR = ROOT / "video" / "logs" / "logs_summary"
OUT_CSV = ROOT / "eval" / "result" / "primary.csv"
THRESHOLDS = (0.3, 0.5, 0.75)
GT_PATTERN = "vid*_true_summary.csv"
PRED_PATTERN = "vid*_summary_log.csv"

print(f"GT:   {GT_DIR}")
print(f"PRED: {PRED_DIR}")
print(f"OUT:  {OUT_CSV}")

GT:   c:\Users\Cent\Desktop\thesisRun\thesis\src\eval\gt_summary
PRED: c:\Users\Cent\Desktop\thesisRun\thesis\src\video\logs\logs_summary
OUT:  c:\Users\Cent\Desktop\thesisRun\thesis\src\eval\result\primary.csv


In [2]:
def _vidnum(name):
    m = re.search(r"vid(\d+)", name)
    return int(m.group(1)) if m else None


def pair_tiou(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]) + 1)
    union = (a[1] - a[0] + 1) + (b[1] - b[0] + 1) - inter
    return inter / union if union > 0 else 0.0


def score_video(gt_path, pred_path, thresholds=THRESHOLDS):
    g = pd.read_csv(gt_path)[["frame_start", "frame_end"]].values.tolist()
    p = pd.read_csv(pred_path)[["frame_start", "frame_end"]].values.tolist()
    n_gt, n_pred = len(g), len(p)
    out = {"n_gt": n_gt, "n_pred": n_pred}
    if n_gt == 0 or n_pred == 0:
        for t in thresholds:
            out[f"P@{t:g}"], out[f"R@{t:g}"], out[f"F1@{t:g}"] = 0.0, 0.0, 0.0
        out["mean_F1"], out["n_match"] = 0.0, 0
        return out
    M = np.array([[pair_tiou(a, b) for b in p] for a in g])
    row, col = linear_sum_assignment(-M)
    matched = sorted([float(M[r, c]) for r, c in zip(row.tolist(), col.tolist()) if M[r, c] > 0], reverse=True)
    out["n_match"] = len(matched)
    f1s = []
    for t in thresholds:
        tp = sum(1 for v in matched if v >= t)
        P = tp / n_pred if n_pred else 0.0
        R = tp / n_gt if n_gt else 0.0
        F = 2 * P * R / (P + R) if (P + R) else 0.0
        out[f"P@{t:g}"], out[f"R@{t:g}"], out[f"F1@{t:g}"] = P, R, F
        f1s.append(F)
    out["mean_F1"] = float(np.mean(f1s))
    return out


def discover(gt_dir, pred_dir):
    gt = {n: f for f in gt_dir.glob(GT_PATTERN) if (n := _vidnum(f.name)) is not None}
    pr = {n: f for f in pred_dir.glob(PRED_PATTERN) if (n := _vidnum(f.name)) is not None}
    pairs = [(v, gt[v], pr[v]) for v in sorted(set(gt) & set(pr))]
    for v in sorted(set(gt) - set(pr)):
        print(f"skip vid{v:02d}: GT without prediction ({gt[v].name})")
    for v in sorted(set(pr) - set(gt)):
        print(f"skip vid{v:02d}: prediction without GT ({pr[v].name})")
    return pairs

In [3]:
pairs = discover(GT_DIR, PRED_DIR)
print(f"scoring {len(pairs)} videos")

rows = []
for v, gp, pp in pairs:
    rows.append({"video": f"vid{v:02d}", **score_video(gp, pp, THRESHOLDS)})
df = pd.DataFrame(rows)

cols = ["video", "n_gt", "n_pred"] + [c for t in THRESHOLDS for c in (f"P@{t:g}", f"R@{t:g}", f"F1@{t:g}")] + ["mean_F1", "n_match"]
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
df[cols]

scoring 30 videos


,video,n_gt,n_pred,P@0.3,R@0.3,F1@0.3,P@0.5,R@0.5,F1@0.5,P@0.75,R@0.75,F1@0.75,mean_F1,n_match
0,vid01,4,8,0.500000,1.000000,0.666667,0.500000,1.000000,0.666667,0.500000,1.000000,0.666667,0.666667,4
1,vid02,3,3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3
2,vid03,1,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1
3,vid04,41,54,0.740741,0.975610,0.842105,0.703704,0.926829,0.800000,0.666667,0.878049,0.757895,0.800000,41
4,vid05,12,24,0.333333,0.666667,0.444444,0.250000,0.500000,0.333333,0.208333,0.416667,0.277778,0.351852,11
5,vid07,3,3,1.000000,1.000000,1.000000,0.666667,0.666667,0.666667,0.666667,0.666667,0.666667,0.777778,3
6,vid08,14,24,0.583333,1.000000,0.736842,0.541667,0.928571,0.684211,0.458333,0.785714,0.578947,0.666667,14
7,vid09,15,20,0.700000,0.933333,0.800000,0.650000,0.866667,0.742857,0.500000,0.666667,0.571429,0.704762,14
8,vid11,6,8,0.750000,1.000000,0.857143,0.500000,0.666667,0.571429,0.375000,0.500000,0.428571,0.619048,6
9,vid12,7,12,0.583333,1.000000,0.736842,0.583333,1.000000,0.736842,0.500000,0.857143,0.631579,0.701754,7


In [4]:
print(f"=== DATASET (mean +- sample std, n={len(df)}) ===")
for c in cols[3:]:
    print(f"{c:10s} {df[c].mean():.2f} +- {df[c].std(ddof=1):.2f}")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}")

=== DATASET (mean +- sample std, n=30) ===
P@0.3      0.7258 +- 0.2268
R@0.3      0.9416 +- 0.0895
F1@0.3     0.8022 +- 0.1706
P@0.5      0.6820 +- 0.2364
R@0.5      0.8818 +- 0.1401
F1@0.5     0.7524 +- 0.1896
P@0.75     0.6075 +- 0.2596
R@0.75     0.7841 +- 0.2041
F1@0.75    0.6687 +- 0.2251
mean_F1    0.7411 +- 0.1833
n_match    7.6333 +- 7.5862
wrote c:\Users\Cent\Desktop\thesisRun\thesis\src\eval\result\primary.csv
